# 1.3 Linear Regression

#### 코드 참조
[1] https://github.com/rickiepark/machine-learning-with-python-cookbook
[2] https://scikit-learn.org/stable/modules/linear_model.html
[3] https://numpy.org/doc/stable/reference/routines.linalg.html
[4] Hastie et al., *The Elements of Statistical Learning*, Ch. 3
[5] Dr. Jaewook Lee's Lecture Notes

## 1.3.1 보스턴 주택가격 데이터 불러오기

`sklearn.datasets.load_boston()`은 데이터에 포함된 변수 `B`의 윤리적 문제로
scikit-learn 1.2부터 **삭제**되었습니다. 따라서 원본 CSV를 직접 읽어 사용합니다.
(연구/논문 목적이라면 `fetch_california_housing()`이나 Ames housing을 권장합니다.)

In [ ]:
# 라이브러리를 임포트합니다.
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

np.set_printoptions(precision=3, suppress=True)
pd.set_option('display.precision', 3)

In [ ]:
# 데이터를 읽어옵니다(506개 행, 13개 특성 + 타깃).
url = 'https://raw.githubusercontent.com/selva86/datasets/master/BostonHousing.csv'
df = pd.read_csv(url)
df.head()

| 변수 | 설명 |
|---|---|
| `crim` | 지역별 1인당 범죄율 |
| `zn` | 25,000 sq.ft. 이상 주거지 비율 |
| `indus` | 비소매 상업지구 비율 |
| `chas` | 찰스강 인접 여부(1/0) |
| `nox` | 질소산화물 농도 |
| `rm` | 주택당 평균 방 개수 |
| `age` | 1940년 이전에 지어진 주택 비율 |
| `dis` | 5개 고용센터까지의 가중 거리 |
| `rad` | 방사형 고속도로 접근성 지수 |
| `tax` | 재산세율 |
| `ptratio` | 학생/교사 비율 |
| `b` | 인구 구성 기반 파생변수 (아래 주의 참고) |
| `lstat` | 저소득층 인구 비율 |
| `medv` | **타깃**: 주택가격 중앙값(단위 $1,000) |

In [ ]:
# 데이터 크기와 결측치를 확인합니다.
print(df.shape)
df.isna().sum().sum()

In [ ]:
# 기술통계량
df.describe().T[['mean', 'std', 'min', 'max']]

In [ ]:
# 타깃 변수의 분포 (50에서 잘린 것처럼 보이는 값들을 확인해 보세요)
plt.hist(df['medv'], bins=40)
plt.xlabel('medv ($1000s)')
plt.ylabel('count')
plt.show()

In [ ]:
# 타깃과의 상관계수를 절댓값 기준으로 정렬합니다.
corr = df.corr()['medv'].drop('medv')
corr.reindex(corr.abs().sort_values(ascending=False).index)

In [ ]:
# 상관계수 행렬을 그림으로 봅니다.
plt.figure(figsize=(8, 7))
plt.imshow(df.corr(), cmap='coolwarm', vmin=-1, vmax=1)
plt.xticks(range(df.shape[1]), df.columns, rotation=90)
plt.yticks(range(df.shape[1]), df.columns)
plt.colorbar()
plt.show()

In [ ]:
# 상관이 큰 두 변수와 타깃의 산점도
fig, axes = plt.subplots(1, 2, figsize=(10, 4))
for ax, col in zip(axes, ['rm', 'lstat']):
    ax.scatter(df[col], df['medv'], s=10, alpha=0.5)
    ax.set_xlabel(col)
    ax.set_ylabel('medv')
    ax.grid()
plt.show()

## 1.3.2 단순선형회귀(simple linear regression)

$$\hat{y} = \beta_0 + \beta_1 x, \qquad
\hat{\boldsymbol\beta} = \arg\min_{\boldsymbol\beta} \sum_{i=1}^{n}\bigl(y_i - \beta_0 - \beta_1 x_i\bigr)^2$$

In [ ]:
# 방 개수(rm)로 주택가격(medv)을 설명해 봅니다.
x = df['rm'].values
y = df['medv'].values

n = len(y)
n

In [ ]:
# 절편(intercept)을 위해 1로 채워진 열을 추가합니다. -> 설계행렬(design matrix)
X = np.column_stack([np.ones(n), x])
X[:5]

정규방정식(normal equation)

$$X^{\top}X\,\hat{\boldsymbol\beta} = X^{\top}\mathbf{y}
\;\;\Longrightarrow\;\;
\hat{\boldsymbol\beta} = (X^{\top}X)^{-1}X^{\top}\mathbf{y}$$

In [ ]:
# 정규방정식을 직접 풀어봅니다.
beta = np.linalg.inv(X.T @ X) @ X.T @ y
beta

In [ ]:
# 공식으로도 확인합니다: beta1 = Cov(x, y) / Var(x)
beta1 = np.sum((x - x.mean()) * (y - y.mean())) / np.sum((x - x.mean()) ** 2)
beta0 = y.mean() - beta1 * x.mean()

print(beta0, beta1)

In [ ]:
# 회귀직선을 그립니다.
y_hat = X @ beta

plt.scatter(x, y, s=10, alpha=0.4, label='data')
plt.plot(np.sort(x), beta[0] + beta[1] * np.sort(x), 'r', label='OLS fit')
plt.xlabel('rm')
plt.ylabel('medv')
plt.grid()
plt.legend()
plt.show()

$$R^2 = 1 - \frac{\text{RSS}}{\text{TSS}}
= 1 - \frac{\sum (y_i - \hat{y}_i)^2}{\sum (y_i - \bar{y})^2}$$

In [ ]:
# 잔차와 성능 지표
resid = y - y_hat

RSS = np.sum(resid ** 2)
TSS = np.sum((y - y.mean()) ** 2)
R2 = 1 - RSS / TSS
RMSE = np.sqrt(RSS / n)

print(f'RSS = {RSS:.2f}, R2 = {R2:.4f}, RMSE = {RMSE:.3f}')

In [ ]:
# 단순회귀에서 R^2는 상관계수의 제곱과 같습니다.
np.corrcoef(x, y)[0, 1] ** 2

In [ ]:
# 잔차의 성질: 합이 0이고, 설명변수와 직교합니다.
print(resid.sum())
print(X.T @ resid)

## 1.3.3 최소제곱해를 구하는 여러 방법

정규방정식은 $X^{\top}X$의 조건수가 원래의 제곱이 되어 수치적으로 불안정합니다.
실제로는 QR 분해나 SVD 기반 방법을 사용합니다(1.2 참고).

In [ ]:
# (1) 역행렬 직접 계산 - 가장 느리고 불안정
b1 = np.linalg.inv(X.T @ X) @ X.T @ y

# (2) 선형시스템 풀이 - 역행렬을 만들지 않음
b2 = np.linalg.solve(X.T @ X, X.T @ y)

# (3) 최소제곱 전용 함수
b3 = np.linalg.lstsq(X, y, rcond=None)[0]

# (4) 유사역행렬(SVD 기반)
b4 = np.linalg.pinv(X) @ y

np.vstack([b1, b2, b3, b4])

In [ ]:
# (5) QR 분해: X = QR -> R beta = Q^T y
Q, R = np.linalg.qr(X)
b5 = np.linalg.solve(R, Q.T @ y)
b5

In [ ]:
# (6) SVD를 직접 사용: beta = V diag(1/s) U^T y
U, s, Vt = np.linalg.svd(X, full_matrices=False)
b6 = Vt.T @ np.diag(1 / s) @ U.T @ y
b6

In [ ]:
# 사영행렬(hat matrix) H = X(X^T X)^-1 X^T -> y_hat = H y
H = X @ np.linalg.inv(X.T @ X) @ X.T

print(np.allclose(H @ y, X @ beta))
print(np.allclose(H @ H, H))          # 멱등(idempotent)
print(np.trace(H))                     # trace = 모수의 개수

## 1.3.4 다중선형회귀(multiple linear regression)

In [ ]:
# 13개 특성을 모두 사용합니다.
feature_names = [c for c in df.columns if c != 'medv']

X_raw = df[feature_names].values
y = df['medv'].values

X = np.column_stack([np.ones(len(y)), X_raw])
X.shape

In [ ]:
beta = np.linalg.lstsq(X, y, rcond=None)[0]

pd.Series(beta, index=['intercept'] + feature_names)

In [ ]:
# 예측과 성능 지표
y_hat = X @ beta
resid = y - y_hat

n, p = X_raw.shape                      # p = 특성 개수(절편 제외)
RSS = np.sum(resid ** 2)
TSS = np.sum((y - y.mean()) ** 2)

R2 = 1 - RSS / TSS
R2_adj = 1 - (RSS / (n - p - 1)) / (TSS / (n - 1))
RMSE = np.sqrt(RSS / n)

print(f'R2 = {R2:.4f}, adjusted R2 = {R2_adj:.4f}, RMSE = {RMSE:.3f}')

계수의 크기를 서로 비교하려면 단위를 맞춰야 합니다(**표준화**).
표준화된 계수는 "해당 변수가 1 표준편차 증가할 때 타깃의 변화량"으로 해석합니다.

In [ ]:
# 표준화(standardization)
X_std = (X_raw - X_raw.mean(axis=0)) / X_raw.std(axis=0)
Xs = np.column_stack([np.ones(len(y)), X_std])

beta_std = np.linalg.lstsq(Xs, y, rcond=None)[0]

coef_std = pd.Series(beta_std[1:], index=feature_names)
coef_std.reindex(coef_std.abs().sort_values(ascending=False).index)

In [ ]:
# 표준화해도 예측값과 R^2는 바뀌지 않습니다(절편만 평균으로 이동).
print(beta_std[0], y.mean())
np.allclose(Xs @ beta_std, X @ beta)

In [ ]:
# 표준화 계수를 막대그래프로 봅니다.
coef_std.plot(kind='barh', figsize=(6, 5))
plt.axvline(0, color='k', lw=0.8)
plt.xlabel('standardized coefficient')
plt.grid(axis='x')
plt.show()

In [ ]:
# 실제값 vs 예측값
plt.figure(figsize=(5, 5))
plt.scatter(y, y_hat, s=10, alpha=0.5)
plt.plot([0, 50], [0, 50], 'r--')
plt.xlabel('actual medv')
plt.ylabel('predicted medv')
plt.grid()
plt.show()

## 1.3.5 계수의 표준오차와 유의성 검정

$$\hat{\sigma}^2 = \frac{\text{RSS}}{n - p - 1}, \qquad
\operatorname{Var}(\hat{\boldsymbol\beta}) = \hat{\sigma}^2 (X^{\top}X)^{-1}, \qquad
t_j = \frac{\hat{\beta}_j}{\operatorname{SE}(\hat{\beta}_j)}$$

In [ ]:
from scipy import stats

# 잔차분산 추정
sigma2 = RSS / (n - p - 1)

# 계수의 공분산행렬
cov_beta = sigma2 * np.linalg.inv(X.T @ X)
se = np.sqrt(np.diag(cov_beta))

se[:5]

In [ ]:
# t 통계량과 p 값
t_stat = beta / se
p_val = 2 * (1 - stats.t.cdf(np.abs(t_stat), df=n - p - 1))

summary = pd.DataFrame({
    'coef': beta,
    'std err': se,
    't': t_stat,
    'P>|t|': p_val,
}, index=['intercept'] + feature_names)

summary

In [ ]:
# 유의하지 않은 변수(p > 0.05)
summary[summary['P>|t|'] > 0.05]

In [ ]:
# 95% 신뢰구간
t_crit = stats.t.ppf(0.975, df=n - p - 1)

summary['CI 2.5%'] = beta - t_crit * se
summary['CI 97.5%'] = beta + t_crit * se
summary[['coef', 'CI 2.5%', 'CI 97.5%']]

In [ ]:
# 모형 전체의 유의성(F 검정)
F = (TSS - RSS) / p / (RSS / (n - p - 1))
p_F = 1 - stats.f.cdf(F, p, n - p - 1)

print(f'F = {F:.2f}, p-value = {p_F:.3e}')

## 1.3.6 잔차 분석(residual analysis)

In [ ]:
# (1) 잔차 vs 예측값: 패턴이 없어야 합니다.
plt.scatter(y_hat, resid, s=10, alpha=0.5)
plt.axhline(0, color='r', lw=1)
plt.xlabel('fitted value')
plt.ylabel('residual')
plt.grid()
plt.show()

In [ ]:
# (2) Q-Q plot: 잔차의 정규성을 확인합니다.
stats.probplot(resid, dist='norm', plot=plt)
plt.grid()
plt.show()

In [ ]:
# (3) 잔차의 히스토그램
plt.hist(resid, bins=40)
plt.xlabel('residual')
plt.show()

In [ ]:
# 정규성 검정
stats.shapiro(resid)

In [ ]:
# 표준화잔차가 큰 관측치(이상치 후보)
std_resid = resid / np.sqrt(sigma2)
idx = np.argsort(np.abs(std_resid))[::-1][:5]

df.iloc[idx].assign(std_resid=std_resid[idx])[['rm', 'lstat', 'medv', 'std_resid']]

In [ ]:
# 지렛대(leverage): hat matrix의 대각원소
H = X @ np.linalg.inv(X.T @ X) @ X.T
lev = np.diag(H)

plt.scatter(lev, std_resid, s=10, alpha=0.5)
plt.axhline(0, color='r', lw=1)
plt.axvline(2 * (p + 1) / n, color='g', ls='--')   # 경험적 기준선
plt.xlabel('leverage')
plt.ylabel('standardized residual')
plt.grid()
plt.show()

## 1.3.7 다중공선성(multicollinearity)

설명변수끼리 상관이 높으면 $X^{\top}X$가 특이행렬에 가까워져 계수 추정이 불안정해집니다.
$$\text{VIF}_j = \frac{1}{1 - R_j^2}$$
($R_j^2$는 $j$번째 변수를 나머지 변수들로 회귀했을 때의 결정계수)

In [ ]:
# 조건수: 1.2의 SVD와 연결됩니다.
print('cond(X_std)   =', np.linalg.cond(X_std))
print('cond(X^T X)   =', np.linalg.cond(X_std.T @ X_std))

In [ ]:
# 특이값을 직접 확인합니다.
np.linalg.svd(X_std, compute_uv=False)

In [ ]:
# VIF를 계산합니다.
def vif(X, j):
    Xj = np.delete(X, j, axis=1)
    Xj = np.column_stack([np.ones(len(X)), Xj])
    beta_j = np.linalg.lstsq(Xj, X[:, j], rcond=None)[0]
    r = X[:, j] - Xj @ beta_j
    R2_j = 1 - np.sum(r ** 2) / np.sum((X[:, j] - X[:, j].mean()) ** 2)
    return 1 / (1 - R2_j)

vifs = pd.Series([vif(X_std, j) for j in range(p)], index=feature_names)
vifs.sort_values(ascending=False)

In [ ]:
# VIF가 큰 변수들은 서로 상관이 높습니다.
df[['rad', 'tax', 'nox', 'indus', 'dis']].corr()

In [ ]:
# 상관이 높은 변수를 제거하면 계수가 크게 바뀝니다.
keep = [f for f in feature_names if f != 'tax']
Xk = np.column_stack([np.ones(n), df[keep].values])
beta_k = np.linalg.lstsq(Xk, y, rcond=None)[0]

pd.DataFrame({
    'with tax': pd.Series(beta[1:], index=feature_names),
    'without tax': pd.Series(beta_k[1:], index=keep),
})

## 1.3.8 경사하강법(gradient descent)으로 풀기

$$J(\boldsymbol\beta) = \frac{1}{2n}\|X\boldsymbol\beta - \mathbf{y}\|^2, \qquad
\nabla J = \frac{1}{n}X^{\top}(X\boldsymbol\beta - \mathbf{y}), \qquad
\boldsymbol\beta \leftarrow \boldsymbol\beta - \eta \nabla J$$

경사하강법은 반드시 **표준화된 특성**에 적용해야 수렴이 안정적입니다.

In [ ]:
def gradient_descent(X, y, lr=0.1, n_iter=500):
    n = len(y)
    beta = np.zeros(X.shape[1])
    history = []
    for _ in range(n_iter):
        grad = X.T @ (X @ beta - y) / n
        beta -= lr * grad
        history.append(np.mean((X @ beta - y) ** 2) / 2)
    return beta, np.array(history)

In [ ]:
beta_gd, hist = gradient_descent(Xs, y, lr=0.1, n_iter=500)

# 정규방정식 해와 비교합니다.
pd.DataFrame({'gradient descent': beta_gd, 'closed form': beta_std},
             index=['intercept'] + feature_names)

In [ ]:
# 손실 곡선
plt.plot(hist)
plt.xlabel('iteration')
plt.ylabel('MSE / 2')
plt.yscale('log')
plt.grid()
plt.show()

In [ ]:
# 학습률에 따른 수렴 속도 비교
for lr in [0.01, 0.05, 0.1, 0.5]:
    _, h = gradient_descent(Xs, y, lr=lr, n_iter=300)
    plt.plot(h, label=f'lr = {lr}')

plt.xlabel('iteration')
plt.ylabel('MSE / 2')
plt.yscale('log')
plt.legend()
plt.grid()
plt.show()

In [ ]:
# 표준화하지 않으면 같은 학습률에서 발산합니다. -> 아주 작은 학습률을 써야 합니다.
X_noscale = np.column_stack([np.ones(n), X_raw])
_, h_bad = gradient_descent(X_noscale, y, lr=1e-6, n_iter=300)

h_bad[-1]     # 아주 작은 학습률에서도 수렴이 매우 느립니다.

## 1.3.9 훈련/테스트 분할과 과적합

In [ ]:
# 데이터를 훈련/테스트로 나눕니다.
rng = np.random.default_rng(42)
idx = rng.permutation(n)
n_train = int(n * 0.7)

tr, te = idx[:n_train], idx[n_train:]
print(len(tr), len(te))

In [ ]:
# 훈련 데이터의 평균/표준편차로만 표준화합니다(데이터 누수 방지).
mu, sd = X_raw[tr].mean(axis=0), X_raw[tr].std(axis=0)

Xtr = np.column_stack([np.ones(len(tr)), (X_raw[tr] - mu) / sd])
Xte = np.column_stack([np.ones(len(te)), (X_raw[te] - mu) / sd])
ytr, yte = y[tr], y[te]

In [ ]:
def rmse(y_true, y_pred):
    return np.sqrt(np.mean((y_true - y_pred) ** 2))

b = np.linalg.lstsq(Xtr, ytr, rcond=None)[0]

print(f'train RMSE = {rmse(ytr, Xtr @ b):.3f}')
print(f'test  RMSE = {rmse(yte, Xte @ b):.3f}')

In [ ]:
# 다항 특성을 늘려가며 과적합을 관찰합니다(lstat 하나만 사용).
xtr, xte = df['lstat'].values[tr], df['lstat'].values[te]

train_err, test_err = [], []
degrees = range(1, 16)

for d in degrees:
    Ptr = np.vander(xtr, d + 1)
    Pte = np.vander(xte, d + 1)
    bd = np.linalg.lstsq(Ptr, ytr, rcond=None)[0]
    train_err.append(rmse(ytr, Ptr @ bd))
    test_err.append(rmse(yte, Pte @ bd))

plt.plot(degrees, train_err, 'o-', label='train')
plt.plot(degrees, test_err, 's-', label='test')
plt.xlabel('polynomial degree')
plt.ylabel('RMSE')
plt.legend()
plt.grid()
plt.show()

In [ ]:
# 차수가 높아지면 곡선이 데이터를 과하게 따라갑니다.
grid = np.linspace(xtr.min(), xtr.max(), 200)

plt.figure()
plt.scatter(xtr, ytr, s=10, alpha=0.4)
for d in [1, 2, 12]:
    bd = np.linalg.lstsq(np.vander(xtr, d + 1), ytr, rcond=None)[0]
    plt.plot(grid, np.vander(grid, d + 1) @ bd, label=f'degree {d}')

plt.ylim(0, 60)
plt.xlabel('lstat')
plt.ylabel('medv')
plt.legend()
plt.grid()
plt.show()

## 1.3.10 정규화: 릿지(Ridge)와 라쏘(Lasso)

$$\hat{\boldsymbol\beta}^{\text{ridge}} = \arg\min_{\boldsymbol\beta}
\|\mathbf{y} - X\boldsymbol\beta\|^2 + \lambda\|\boldsymbol\beta\|_2^2
= (X^{\top}X + \lambda I)^{-1}X^{\top}\mathbf{y}$$

$X^{\top}X$에 $\lambda I$를 더하므로 **고유값이 모두 $\lambda$만큼 커져** 항상 역행렬이 존재합니다.

In [ ]:
def ridge(X, y, lam):
    # 절편에는 벌점을 주지 않습니다.
    I = np.eye(X.shape[1])
    I[0, 0] = 0
    return np.linalg.solve(X.T @ X + lam * I, X.T @ y)

In [ ]:
# lambda를 키우면 계수가 0쪽으로 축소(shrinkage)됩니다.
lams = np.logspace(-2, 4, 60)
paths = np.array([ridge(Xtr, ytr, l)[1:] for l in lams])

for j, name in enumerate(feature_names):
    plt.plot(lams, paths[:, j], label=name)

plt.xscale('log')
plt.xlabel('lambda')
plt.ylabel('coefficient')
plt.legend(fontsize=7, ncol=2)
plt.grid()
plt.show()

In [ ]:
# 테스트 오차가 가장 작은 lambda를 찾습니다.
errs = [rmse(yte, Xte @ ridge(Xtr, ytr, l)) for l in lams]

plt.semilogx(lams, errs)
plt.xlabel('lambda')
plt.ylabel('test RMSE')
plt.grid()
plt.show()

lams[int(np.argmin(errs))], min(errs)

### SVD로 본 릿지 회귀
$X = U\Sigma V^{\top}$ 일 때
$$X\hat{\boldsymbol\beta}^{\text{ridge}}
= \sum_j \mathbf{u}_j \frac{\sigma_j^2}{\sigma_j^2 + \lambda}\,\mathbf{u}_j^{\top}\mathbf{y}$$
즉 **작은 특이값 방향일수록 더 강하게 축소**됩니다.

In [ ]:
U, s, Vt = np.linalg.svd(Xtr[:, 1:], full_matrices=False)

for lam in [1, 10, 100, 1000]:
    plt.plot(s ** 2 / (s ** 2 + lam), 'o-', label=f'lambda = {lam}')

plt.xlabel('component index')
plt.ylabel('shrinkage factor')
plt.legend()
plt.grid()
plt.show()

In [ ]:
# 유효 자유도(effective degrees of freedom) = sum of shrinkage factors
for lam in [0.1, 1, 10, 100, 1000]:
    print(f'lambda = {lam:7.1f}  df = {np.sum(s ** 2 / (s ** 2 + lam)):.2f}')

In [ ]:
# 라쏘(Lasso)는 L1 벌점을 사용해 일부 계수를 정확히 0으로 만듭니다.
from sklearn.linear_model import Lasso

for a in [0.01, 0.1, 1.0]:
    m = Lasso(alpha=a, max_iter=10000).fit(Xtr[:, 1:], ytr)
    print(f'alpha = {a:5.2f}  nonzero = {np.sum(m.coef_ != 0):2d}/13  '
          f'test RMSE = {rmse(yte, m.predict(Xte[:, 1:])):.3f}')

In [ ]:
# 라쏘가 남긴 변수들
m = Lasso(alpha=0.1, max_iter=10000).fit(Xtr[:, 1:], ytr)
pd.Series(m.coef_, index=feature_names)[m.coef_ != 0]

### 부록 A: scikit-learn으로 같은 결과 확인하기

In [ ]:
from sklearn.linear_model import LinearRegression, Ridge
from sklearn.metrics import mean_squared_error, r2_score

lr = LinearRegression().fit(X_raw, y)

print(lr.intercept_)
print(lr.coef_)

In [ ]:
# 직접 계산한 계수와 비교합니다.
np.allclose(np.r_[lr.intercept_, lr.coef_], beta)

In [ ]:
print('R2   =', r2_score(y, lr.predict(X_raw)))
print('RMSE =', np.sqrt(mean_squared_error(y, lr.predict(X_raw))))

In [ ]:
# 릿지도 동일한지 확인합니다(sklearn은 절편에 벌점을 주지 않습니다).
lam = 10.0
sk = Ridge(alpha=lam).fit(Xtr[:, 1:], ytr)
ours = ridge(Xtr, ytr, lam)

print(np.allclose(sk.coef_, ours[1:]))
print(np.allclose(sk.intercept_, ours[0]))

In [ ]:
# 교차검증으로 lambda를 고릅니다.
from sklearn.linear_model import RidgeCV

cv = RidgeCV(alphas=np.logspace(-2, 4, 60)).fit(Xtr[:, 1:], ytr)
cv.alpha_

### 부록 B: 타깃 변환과 비선형 항

In [ ]:
# 가격은 오른쪽으로 치우쳐 있으므로 로그 변환이 도움이 될 수 있습니다.
b_log = np.linalg.lstsq(Xtr, np.log(ytr), rcond=None)[0]
pred = np.exp(Xte @ b_log)

print(f'original scale : test RMSE = {rmse(yte, Xte @ b):.3f}')
print(f'log target     : test RMSE = {rmse(yte, pred):.3f}')

In [ ]:
# lstat의 로그 항을 추가하면 성능이 개선됩니다.
def build(idx_):
    Z = np.column_stack([(X_raw[idx_] - mu) / sd,
                         np.log(df['lstat'].values[idx_])])
    return np.column_stack([np.ones(len(idx_)), Z])

Ztr, Zte = build(tr), build(te)
b_z = np.linalg.lstsq(Ztr, ytr, rcond=None)[0]

print(f'without log(lstat) : test RMSE = {rmse(yte, Xte @ b):.3f}')
print(f'with    log(lstat) : test RMSE = {rmse(yte, Zte @ b_z):.3f}')

In [ ]:
# 상호작용 항(interaction term) 예시: rm * lstat
inter_tr = (X_raw[tr, 5] * X_raw[tr, 12]).reshape(-1, 1)
inter_te = (X_raw[te, 5] * X_raw[te, 12]).reshape(-1, 1)

Itr = np.column_stack([Xtr, (inter_tr - inter_tr.mean()) / inter_tr.std()])
Ite = np.column_stack([Xte, (inter_te - inter_tr.mean()) / inter_tr.std()])

b_i = np.linalg.lstsq(Itr, ytr, rcond=None)[0]
print(f'with interaction : test RMSE = {rmse(yte, Ite @ b_i):.3f}')

### 연습문제

1. `rm` 하나만 쓴 단순회귀와 13개 변수를 모두 쓴 다중회귀의 테스트 RMSE를 비교해 보세요.
2. p 값이 0.05보다 큰 변수를 모두 제거한 뒤 adjusted $R^2$가 어떻게 변하는지 확인해 보세요.
3. `medv == 50`인 관측치(상한에서 잘린 값)를 제외하고 다시 적합하면 계수가 어떻게 달라지나요?
4. 릿지의 shrinkage factor $\sigma_j^2/(\sigma_j^2+\lambda)$를 이용해 유효 자유도가 13에 가까워지는 $\lambda$를 찾아보세요.
5. K-겹 교차검증(K=5)을 `numpy`만으로 직접 구현해 릿지의 $\lambda$를 선택해 보세요.
6. `b` 변수를 제외하고 모형을 다시 적합했을 때 예측 성능이 실질적으로 달라지는지 확인해 보세요.

#### 데이터에 대한 주의
이 데이터의 `b` 변수는 지역의 인구 구성을 이용해 만들어진 변수로, 주택가격과의 관계를
인과적으로 해석해서는 안 됩니다. scikit-learn이 이 데이터셋을 삭제한 이유이기도 합니다.
회귀분석 실습용으로만 사용하고, 변수 해석이 필요한 분석에는 다른 데이터를 사용하세요.